In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor
from openbabel import openbabel
from openpyxl import Workbook
import time
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob
from typing import Dict, List  # 中文注释：用于类型注解的标准库
from rdkit import Chem        # 中文注释：导入RDKit的Chem模块以进行分子与原子操作
import os                     # 中文注释：用于处理文件路径与目录
import json                   # 中文注释：用于将列表保存为JSON文件以及从JSON恢复为列表
import re                     # 中文注释：用于安全地清理文件名中可能的非法字符

In [ ]:
# 设定 sobtop 所在的目录
sobtop_directory = result["sobtop_home"]
print(sobtop_directory)

In [ ]:
# 聚合物键长、键角仍沿用 GAFF 力场参数；本单元只处理 MOL2 兼容性并可靠调用 Sobtop。
def _make_mol2_sobtop_compatible(mol2_path):
    """
    功能目的：
        直接修改生成的 MOL2，删除 Sobtop/atomtype 无法识别的 UNITY_ATOM_ATTR 段。

    输入参数：
        mol2_path: 待交给 Sobtop 的 MOL2 文件路径。

    返回值：
        bool；实际删除过目标段时返回 True，文件原本已兼容时返回 False。

    关键流程：
        按 @<TRIPOS> 段落边界删除完整 UNITY_ATOM_ATTR 段；MOLECULE、ATOM、
        BOND 等其他段逐行保持不变。处理是幂等的，可安全重复调用。

    可能报错或边界情况：
        文件不存在、为空、缺少 Sobtop 所需基本段，或写回后仍残留目标段时抛出异常。
    """
    source_path = os.path.abspath(mol2_path)
    if not os.path.isfile(source_path):
        raise FileNotFoundError(f"Sobtop 输入 MOL2 不存在：{source_path}")
    if os.path.getsize(source_path) == 0:
        raise ValueError(f"Sobtop 输入 MOL2 为空：{source_path}")

    with open(source_path, "r", encoding="utf-8") as handle:
        source_lines = handle.readlines()

    marker = "@<TRIPOS>UNITY_ATOM_ATTR"
    required_sections = {
        "@<TRIPOS>MOLECULE",
        "@<TRIPOS>ATOM",
        "@<TRIPOS>BOND",
    }
    sections_before = {
        line.strip().upper()
        for line in source_lines
        if line.strip().upper().startswith("@<TRIPOS>")
    }
    missing_sections = sorted(required_sections - sections_before)
    if missing_sections:
        raise ValueError(
            f"MOL2 缺少 Sobtop 必需段：{missing_sections}；文件={source_path}"
        )

    cleaned_lines = []
    skipping_target_section = False
    removed_section_count = 0
    for line in source_lines:
        section_name = line.strip().upper()
        if section_name == marker:
            skipping_target_section = True
            removed_section_count += 1
            continue
        if skipping_target_section and section_name.startswith("@<TRIPOS>"):
            skipping_target_section = False
        if not skipping_target_section:
            cleaned_lines.append(line)

    if removed_section_count == 0:
        return False

    sections_after = {
        line.strip().upper()
        for line in cleaned_lines
        if line.strip().upper().startswith("@<TRIPOS>")
    }
    missing_after_cleanup = sorted(required_sections - sections_after)
    if missing_after_cleanup:
        raise ValueError(
            f"MOL2 清理后缺少 Sobtop 必需段：{missing_after_cleanup}；文件={source_path}"
        )

    # 中文注释：按用户要求直接写回原 MOL2，不创建 Sobtop 专用副本。
    with open(source_path, "w", encoding="utf-8") as handle:
        handle.writelines(cleaned_lines)

    with open(source_path, "r", encoding="utf-8") as handle:
        cleaned_text = handle.read()
    if not cleaned_text or marker in cleaned_text.upper():
        raise RuntimeError(f"MOL2 原地清理失败：{source_path}")

    print(
        f"[Sobtop兼容] 已原地删除 {removed_section_count} 个 "
        f"UNITY_ATOM_ATTR 段：{source_path}"
    )
    return True


def _run_sobtop_checked(mol2_path, commands, top_path, itp_path):
    """
    功能目的：
        原地兼容化 MOL2，执行 Sobtop，并立即验证返回码及 TOP/ITP 输出。

    输入参数：
        mol2_path: Sobtop 输入 MOL2 路径。
        commands: 发送给 Sobtop 的交互命令序列。
        top_path: 预期 TOP 输出路径。
        itp_path: 预期 ITP 输出路径。

    返回值：
        None；成功条件是 Sobtop 返回 0 且两个输出文件存在并非空。

    关键流程：
        先修改原 MOL2，再删除可能残留的旧拓扑，随后在原有 Sobtop 工作目录运行。

    可能报错或边界情况：
        Sobtop 不可执行、返回非零或未生成有效输出时立即抛出异常。
    """
    sobtop_executable = os.path.join(sobtop_directory, "sobtop")
    if not os.path.isfile(sobtop_executable) or not os.access(sobtop_executable, os.X_OK):
        raise FileNotFoundError(f"Sobtop 可执行文件不存在或不可执行：{sobtop_executable}")

    _make_mol2_sobtop_compatible(mol2_path)

    # 中文注释：删除旧输出，防止历史文件把本次 Sobtop 失败伪装成成功。
    for output_path in (top_path, itp_path):
        if os.path.isfile(output_path) or os.path.islink(output_path):
            os.remove(output_path)
        elif os.path.isdir(output_path):
            raise IsADirectoryError(f"Sobtop 输出路径被目录占用：{output_path}")

    completed = subprocess.run(
        ["./sobtop", os.path.abspath(mol2_path)],
        cwd=sobtop_directory,
        input=commands,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)

    log_path = f"{itp_path}.sobtop.log"
    with open(log_path, "w", encoding="utf-8") as handle:
        handle.write("===== stdout =====\n")
        handle.write(completed.stdout or "")
        handle.write("\n===== stderr =====\n")
        handle.write(completed.stderr or "")

    if completed.returncode != 0:
        detail = (completed.stderr or completed.stdout or "无详细输出")[-4000:]
        raise RuntimeError(
            f"Sobtop 执行失败，返回码={completed.returncode}，MOL2={mol2_path}\n{detail}"
        )

    missing_outputs = [
        path for path in (top_path, itp_path)
        if not os.path.isfile(path) or os.path.getsize(path) == 0
    ]
    if missing_outputs:
        raise RuntimeError(
            f"Sobtop 未生成非空输出：{missing_outputs}；详细输出见 {log_path}"
        )


def run_sobtop(mol2_path, chg_path, top_path, itp_path):
    """
    功能目的：使用外部 CHG 电荷为聚合物生成 GAFF TOP/ITP。
    输入参数：mol2_path、chg_path、top_path、itp_path 均为对应文件路径。
    返回值：None；成功时生成非空 TOP 与 ITP。
    关键流程：校验 CHG，构造原有交互命令，调用统一的严格 Sobtop 执行器。
    可能报错或边界情况：CHG 缺失/为空、Sobtop 失败或输出缺失时直接抛出异常。
    """
    charge_path = os.path.abspath(chg_path)
    if not os.path.isfile(charge_path) or os.path.getsize(charge_path) == 0:
        raise FileNotFoundError(f"Sobtop 聚合物电荷文件不存在或为空：{charge_path}")

    commands = f"7\n10\n{charge_path}\n0\n1\n2\n4\n{top_path}\n{itp_path}\n0\n"
    _run_sobtop_checked(mol2_path, commands, top_path, itp_path)


In [ ]:
# 仅针对聚合物生成top和itp文件，筛选以polymer开头的聚合物名称
def create_polymer_itp_top():
    for filename in os.listdir('.'):
        if filename.endswith('.pdb') and filename.startswith('polymer'):
            # 从文件名中提取名称
            base_filename = os.path.splitext(filename)[0]
            print(base_filename)
            # mol2_path, chg_path, top_path, itp_path
            mol2_path = os.path.abspath(base_filename + '.mol2')
            chg_path = os.path.abspath(base_filename + '.chg')
            top_path = os.path.abspath(base_filename + '.top')
            itp_path = os.path.abspath(base_filename + '.itp')
            # 调用sobtop生成拓扑文件
            run_sobtop(mol2_path, chg_path, top_path, itp_path)


    # 正则表达式模式匹配[moleculetype]后面的polymer_{name}
    pattern = re.compile(r"^\[ moleculetype \]\s*\n;\s*name\s+nrexcl\s*\npolymer_(\S+)\s+(\d+)", re.MULTILINE)
    
    # 遍历当前目录下的所有文件
    for filename in os.listdir('.'):
        # 检查文件名是否符合条件
        if filename.endswith('.itp') and filename.startswith('polymer'):
            # 读取文件内容
            with open(filename, 'r') as file:
                content = file.read()
    
            # 使用正则表达式查找并替换内容
            new_content = pattern.sub(r"[ moleculetype ]\n; name          nrexcl\n\1       \2", content)
    
            # 如果内容发生了变化，就写回文件
            if new_content != content:
                with open(filename, 'w') as file:
                    file.write(new_content)
    
    print("All matching .itp files have been processed.")


In [ ]:
def main():
    # 读取Excel文件中的离子
    df = pd.read_excel('System_homopolymer.xlsx')

    # 初始化列表来存储索聚合物的名称
    polymer_name = df['Name'].astype(str).tolist()
    
    # 遍历聚合物名称列表，针对列表中的所有聚合物均采取这套处理流程
    for name in polymer_name:
        # 检查名称是否在DataFrame的Name列中
        if name in df['Name'].values:
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")
            # 仅针对聚合物生成top和itp文件，筛选以polymer开头的聚合物名称
            create_polymer_itp_top()
            print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")


In [ ]:
if __name__ == '__main__':
    main()

# 使用基于量子化学计算得到的键长与键角参数替换GAFF默认指认的参数

In [ ]:
'''
给出一段函数，函数的输入值为df，作用为：
如果df没有copolymer_name列，那么在df中新建copolymer_name列，就将Name列的第一个值{name}，以polymer_{name}填入所有copolymer_name列中，然后返回df
'''
import pandas as pd

def ensure_copolymer_name(df: pd.DataFrame) -> pd.DataFrame:
    """
    功能：
    如果 df 中不存在 'copolymer_name' 列，
    则新建一列，并用 Name 列的第一个值填充（格式为 polymer_{name}）。

    参数:
    df (pd.DataFrame): 输入的数据表，要求必须包含 Name 列。

    返回:
    pd.DataFrame: 更新后的数据表
    """
    # 判断是否存在 'copolymer_name' 列
    if 'copolymer_name' not in df.columns:
        # 取 Name 列的第一个值
        first_name = df['Name'].iloc[0]
        # 构造填充值
        fill_value = f"polymer_{first_name}"
        # 新建并填充 copolymer_name 列
        df['copolymer_name'] = fill_value
    
    return df

In [ ]:
df = pd.read_excel('System_homopolymer.xlsx')
# 中文注释：该单元位于 helper 定义之前，因此在这里使用内联过滤，去掉 Excel 空占位行。
df = df.dropna(subset=['Name', 'SMILES']).copy()
for _column in ['Name', 'SMILES']:
    _text_values = df[_column].astype(str).str.strip()
    df = df[(_text_values != '') & (_text_values.str.lower() != 'nan')].copy()
if df.empty:
    raise ValueError("System_homopolymer.xlsx 中没有有效聚合物行：Name 与 SMILES 不能为空。")
df = df.reset_index(drop=True)
ensure_copolymer_name(df)
df.to_excel('System_homopolymer.xlsx', index=None)

In [ ]:
def load_polymer_atom_list(file_path: str) -> List[str]:
    """
    中文功能说明：
        从保存的 {name}_atom_list.json 文件中恢复 polymer_atom_list（Python 列表）

    参数：
        file_path: str
            中文注释：目标 JSON 文件路径（通常由 save_polymer_atom_lists 的返回值中获取）

    返回：
        List[str]
            中文注释：返回原先保存的 polymer_atom_list（字符串列表）
    """
    # 中文注释：基本存在性检查，便于友好报错
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"未找到文件：{file_path}")

    # 中文注释：读取JSON并解析为Python对象
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 中文注释：类型校验，保证读回的数据确为列表
    if not isinstance(data, list):
        raise ValueError(f"文件内容不是列表：{file_path}")

    # 中文注释：进一步校验列表元素是否均为字符串（可根据需要放宽）
    if not all(isinstance(x, str) for x in data):
        raise ValueError(f"列表元素存在非字符串项：{file_path}")

    # 中文注释：返回从JSON中恢复的 polymer_atom_list
    return data

In [ ]:
'''
还缺少直接通过SMILES生成3D结构并且存储在当前目录下
给出一段函数，函数的输入值为df
'''
# --- 使用OpenBabel进行构象搜索与能量优化 ---
def generate_lowest_energy_conformer_openbabel(smiles: str, num_confs: int = 50, forcefield: str = "UFF"):
    """
    根据 SMILES 使用 OpenBabel 进行构象搜索和能量优化，
    返回能量最低构象的索引、能量（kcal/mol）以及3D坐标。
    
    参数：
        smiles (str): 输入的 SMILES 字符串
        num_confs (int): 要生成的构象数量，默认50
        forcefield (str): 力场选择，可选 "MMFF94" 或 "UFF"，默认使用 UFF
        
    返回：
        lowest_conf_index (int): 能量最低构象在 OBMol 中的索引
        lowest_energy (float): 该构象的能量（单位：kcal/mol）
        coordinates (list of tuple): 每个元组为 (原子类型, x, y, z)
    """
    # 1. 创建 OBConversion 对象，并设定输入格式为 SMILES
    obConversion = openbabel.OBConversion()
    obConversion.SetInFormat("smi")
    
    # 2. 从 SMILES 创建 OBMol 对象
    obmol = openbabel.OBMol()
    obConversion.ReadString(obmol, smiles)
    
    # 3. 添加氢原子（确保分子完整性）
    obmol.AddHydrogens()
    
    # 4. 生成初始3D结构（更改部分：增加OBBuilder调用）
    builder = openbabel.OBBuilder()
    builder.Build(obmol)
    
    # 5. 利用 OBConformerSearch 生成多个构象
    cs = openbabel.OBConformerSearch()
    cs.Setup(obmol, num_confs, True)
    cs.Search()
    cs.GetConformers(obmol)
    
    # 6. 选择力场（此处基于UFF力场）
    forcefield = forcefield.upper()
    if forcefield == "MMFF94":
        ff = openbabel.OBForceField.FindForceField("mmff94")
    elif forcefield == "UFF":
        ff = openbabel.OBForceField.FindForceField("uff")
    else:
        raise ValueError("不支持的力场类型，请选择 'MMFF94' 或 'UFF'")
    
    lowest_energy = float('inf')
    lowest_conf_index = None
    nconfs = obmol.NumConformers()
    
    # 7. 遍历所有构象，对每个构象进行能量最小化和能量计算
    for i in range(nconfs):
        obmol.SetConformer(i)
        if not ff.Setup(obmol):
            print(f"构象 {i} 力场设置失败")
            continue
        ff.ConjugateGradients(250, 1.0e-4)
        ff.GetCoordinates(obmol)
        energy = ff.Energy()
        if energy < lowest_energy:
            lowest_energy = energy
            lowest_conf_index = i

    # --- 更改部分：如果最低能构象生成失败，则返回None ---
    if lowest_conf_index is None:
        return None, None, None
    
    atomic_number_to_symbol = {
        1: "H",    2: "He",   3: "Li",   4: "Be",   5: "B",    6: "C",    7: "N",    8: "O",    9: "F",    10: "Ne",
        11: "Na",  12: "Mg",  13: "Al",  14: "Si",  15: "P",   16: "S",   17: "Cl",  18: "Ar",  19: "K",   20: "Ca",
        21: "Sc",  22: "Ti",  23: "V",   24: "Cr",  25: "Mn",  26: "Fe",  27: "Co",  28: "Ni",  29: "Cu",  30: "Zn",
        31: "Ga",  32: "Ge",  33: "As",  34: "Se",  35: "Br",  36: "Kr",  37: "Rb",  38: "Sr",  39: "Y",   40: "Zr",
        41: "Nb",  42: "Mo",  43: "Tc",  44: "Ru",  45: "Rh",  46: "Pd",  47: "Ag",  48: "Cd",  49: "In",  50: "Sn",
        51: "Sb",  52: "Te",  53: "I",   54: "Xe",  55: "Cs",  56: "Ba",  57: "La",  58: "Ce",  59: "Pr",  60: "Nd",
        61: "Pm",  62: "Sm",  63: "Eu",  64: "Gd",  65: "Tb",  66: "Dy",  67: "Ho",  68: "Er",  69: "Tm",  70: "Yb",
        71: "Lu",  72: "Hf",  73: "Ta",  74: "W",   75: "Re",  76: "Os",  77: "Ir",  78: "Pt",  79: "Au",  80: "Hg",
        81: "Tl",  82: "Pb",  83: "Bi",  84: "Po",  85: "At",  86: "Rn",  87: "Fr",  88: "Ra",  89: "Ac",  90: "Th",
        91: "Pa",  92: "U",   93: "Np",  94: "Pu",  95: "Am",  96: "Cm",  97: "Bk",  98: "Cf",  99: "Es", 100: "Fm",
        101: "Md", 102: "No", 103: "Lr", 104: "Rf", 105: "Db", 106: "Sg", 107: "Bh", 108: "Hs", 109: "Mt", 110: "Ds",
        111: "Rg", 112: "Cn", 113: "Nh", 114: "Fl", 115: "Mc", 116: "Lv", 117: "Ts", 118: "Og"
    }

    # 8. 设置 OBMol 到能量最低的构象，并提取3D坐标
    obmol.SetConformer(lowest_conf_index)
    coordinates = []
    # 通过原子序数映射获取纯元素符号
    for atom in pybel.ob.OBMolAtomIter(obmol):
        atomic_num = atom.GetAtomicNum()
        symbol = atomic_number_to_symbol.get(atomic_num, str(atomic_num))  # 如果字典中不存在，则返回数字字符串
        x = atom.GetX()
        y = atom.GetY()
        z = atom.GetZ()
        coordinates.append((symbol, x, y, z))
    
    return lowest_conf_index, lowest_energy, coordinates

In [ ]:
# -*- coding: utf-8 -*-
import os                                     # 中文注释：用于文件路径与写出
import re                                     # 中文注释：用于清洗文件名非法字符
import pandas as pd                           # 中文注释：用于DataFrame行遍历
from typing import List, Tuple, Optional      # 中文注释：类型注解

from rdkit import Chem                        # 中文注释：RDKit核心模块
from rdkit.Chem import AllChem                # 中文注释：3D嵌入与力场优化
from rdkit.Chem import rdmolfiles             # 中文注释：可能包含Mol2写出API
from rdkit.Chem import AddHs, RWMol           # 中文注释：与“不可更改代码块”一致的导入
from rdkit.Geometry import Point3D            # 中文注释：设置3D坐标所需

# 中文注释：尝试导入Open Babel/pybel作为Mol2写出的回退方案（若RDKit无Mol2写出API）
try:
    from openbabel import openbabel, pybel                             # 中文注释：Open Babel的Python高级接口
    _HAVE_PYBEL = True
except Exception:
    _HAVE_PYBEL = False


def _sanitize_filename(name: str) -> str:
    """中文注释：清洗文件名中不合法的字符，避免写出失败"""
    return re.sub(r'[\\/:*?"<>|\s]+', '_', str(name)).strip('_')


def _assign_coords_to_mol(mol: Chem.Mol, coordinates_list: List[Tuple[str, float, float, float]]) -> Chem.Mol:
    """
    中文功能说明：
        将坐标列表写回到给定的 RDKit 分子（原子数需一致），返回带3D构象的分子。
    """
    # 中文注释：安全复制，避免原分子被就地修改
    mol = Chem.Mol(mol)
    # 中文注释：基本一致性校验
    if mol.GetNumAtoms() != len(coordinates_list):
        raise ValueError("坐标数量与分子原子数不一致，无法赋值。")

    # 中文注释：构建一个新的构象并逐原子设置坐标
    conf = Chem.Conformer(mol.GetNumAtoms())
    for idx, (_sym, x, y, z) in enumerate(coordinates_list):
        conf.SetAtomPosition(idx, Point3D(x, y, z))

    # 中文注释：清空旧构象并添加新构象
    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)
    return mol


def _write_mol2(mol: Chem.Mol, out_path: str, title: str) -> None:
    """
    中文功能说明：
        写出mol2文件：优先使用RDKit的Mol2写出API；若不可用则回退到pybel（若已安装）。
    """
    # 中文注释：优先尝试RDKit的Mol2写出（MolToMol2Block）
    if hasattr(rdmolfiles, "MolToMol2Block"):
        mol2_block = rdmolfiles.MolToMol2Block(mol)    # 中文注释：生成mol2文本
        # 中文注释：在mol2文件头部的@<TRIPOS>MOLECULE标题默认来自分子名称，此处不强行改动
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(mol2_block)
        return

    # 中文注释：若RDKit不支持Mol2写出，则尝试用pybel回退（通过SDF中间格式）
    if _HAVE_PYBEL:
        sdf_block = Chem.MolToMolBlock(mol)           # 中文注释：RDKit导出SDF文本（含坐标）
        obmol = pybel.readstring("sdf", sdf_block)    # 中文注释：pybel读取SDF
        obmol.title = title                           # 中文注释：设置标题为Name
        obmol.write("mol2", out_path, overwrite=True) # 中文注释：写出mol2
        return

    # 中文注释：两种方式都不可用时抛出明确异常
    raise RuntimeError("当前环境不支持Mol2写出：RDKit缺少Mol2 API且未检测到pybel。")

In [ ]:
'''
请你修改下列函数，函数的输入值为df，
'''


def _filter_valid_polymer_rows_for_topology(df: pd.DataFrame) -> pd.DataFrame:
    """
    中文功能说明：
        过滤聚合物拓扑生成阶段的有效输入行，避免 Excel 中的空占位行进入 RDKit/OpenBabel。

    输入参数：
        df: 从 System_*.xlsx 读取的原始表格，必须包含 Name 与 SMILES 列。

    返回值：
        仅保留 Name、SMILES 均有效的 DataFrame 副本。

    关键流程：
        1) 检查必要列；
        2) 去掉 Name/SMILES 为 NaN 的行；
        3) 去掉空字符串和字符串形式的 "nan"；
        4) 重置索引，保证后续按行遍历稳定。

    可能报错或边界情况：
        若缺少必要列或没有任何有效聚合物行，则抛出 ValueError，避免静默生成错误拓扑。
    """
    required_columns = {"Name", "SMILES"}
    if not required_columns.issubset(df.columns):
        raise ValueError("输入 df 必须包含列：'Name' 与 'SMILES'。")

    valid_df = df.dropna(subset=["Name", "SMILES"]).copy()
    for column in ["Name", "SMILES"]:
        text_values = valid_df[column].astype(str).str.strip()
        valid_df = valid_df[(text_values != "") & (text_values.str.lower() != "nan")].copy()

    if valid_df.empty:
        raise ValueError("System 表格中没有有效聚合物行：Name 与 SMILES 不能为空。")

    return valid_df.reset_index(drop=True)

def create_monomer_mol2file(df: pd.DataFrame) -> None:
    """
    中文功能说明：
        遍历 df（需包含“Name”“SMILES”列），对每一行分子：
        1) 将虚拟原子“*”替换为氢；
        2) 基于UFF进行构象搜索，获取最低能量构象坐标；
        3) 将坐标回填到分子并在当前目录写出 {Name}.mol2。

    返回：
        None
    """
    # 中文注释：检查必要列
    if not {"Name", "SMILES"}.issubset(df.columns):
        raise ValueError("输入 df 必须包含列：'Name' 与 'SMILES'。")

    # 中文注释：先过滤 Excel 空占位行，避免 NaN 被传入 RDKit。
    valid_df = _filter_valid_polymer_rows_for_topology(df)

    # 中文注释：逐行处理有效聚合物行。
    for row in valid_df.itertuples(index=False):
        # 中文注释：读取名称与SMILES
        name = getattr(row, "Name")
        smiles = getattr(row, "SMILES")

        # 对重复单元加氢，然后获取加氢后的SMILES式中每个元素符号以及对应原子索引。
        mol = Chem.MolFromSmiles(smiles)
        mol_with_h = AddHs(mol)

        # 记录所有原子类型为“*”的原子索引，记为端基索引。
        terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']

        # 创建一个新的可读写分子对象
        rw_mol = RWMol(mol_with_h)

        # 替换端基原子为氢原子
        for idx in terminal_indices:
            rw_mol.ReplaceAtom(idx, Chem.Atom(1))  # 使用原子序号创建氢原子（原子序号为1）
            rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  # 设置为没有隐式氢

        # 更新分子的价态信息
        rw_mol.UpdatePropertyCache(strict=False)

        # 转换回不可变的分子对象
        new_mol = rw_mol.GetMol()

        # 重新添加氢原子以确保分子完整
        new_mol_with_h = AddHs(new_mol)

        # 转换为SMILES字符串
        smiles_with_h = Chem.MolToSmiles(new_mol_with_h)

        # 调用构象搜索函数（基于UFF力场）
        lowest_conf_index, lowest_energy, coordinates_list = generate_lowest_energy_conformer_openbabel(smiles_with_h, num_confs=50, forcefield="UFF")
        # ======================⚠️ 原样代码块结束 ⚠️===========================================================

        # 中文注释：若构象搜索失败则跳过写出
        if lowest_conf_index is None or coordinates_list is None:
            print(f"[跳过] {name}：构象搜索失败。")
            continue

        # 中文注释：将坐标写回到 new_mol_with_h（与 smiles_with_h 一致的原子序）
        try:
            mol3d = _assign_coords_to_mol(new_mol_with_h, coordinates_list)
        except Exception as e:
            print(f"[失败] {name}：坐标赋值失败 -> {e}")
            continue

        # 中文注释：构造输出文件路径（当前目录）
        out_name = _sanitize_filename(name) + ".mol2"
        out_path = os.path.join(".", out_name)

        # 中文注释：尝试写出mol2（RDKit优先，失败则pybel回退）
        try:
            _write_mol2(mol3d, out_path, title=str(name))
            print(f"[完成] 写出 {out_path}")
        except Exception as e:
            print(f"[失败] {name}：Mol2写出失败 -> {e}")
   

In [ ]:
_system_excel_file = "System_homopolymer.xlsx"
df = pd.read_excel(_system_excel_file)
df = _filter_valid_polymer_rows_for_topology(df)
df.to_excel(_system_excel_file, index=False)
create_monomer_mol2file(df) # 创建用H饱和后的单体的mol2文件

In [ ]:
# 单体拓扑使用量子化学 Hessian/电荷信息，并复用前面定义的 MOL2 原地兼容处理。
def run_monomer_sobtop(mol2_path, fchk_path, top_path, itp_path):
    """
    功能目的：
        使用单体 FCHK 中的量子化学信息生成 TOP/ITP。

    输入参数：
        mol2_path: 单体 MOL2 路径。
        fchk_path: Gaussian FCHK 路径。
        top_path: 预期 TOP 输出路径。
        itp_path: 预期 ITP 输出路径。

    返回值：
        None；成功时原 MOL2 已兼容 Sobtop，且生成非空 TOP 与 ITP。

    关键流程：
        校验 FCHK，构造原有 Sobtop 交互命令，复用统一的原地兼容及输出检查逻辑。

    可能报错或边界情况：
        FCHK 缺失/为空、Sobtop 返回非零或输出缺失时直接抛出异常。
    """
    formatted_checkpoint_path = os.path.abspath(fchk_path)
    if not os.path.isfile(formatted_checkpoint_path) or os.path.getsize(formatted_checkpoint_path) == 0:
        raise FileNotFoundError(
            f"Sobtop 单体 FCHK 文件不存在或为空：{formatted_checkpoint_path}"
        )

    commands = (
        f"1\n2\n7\n{formatted_checkpoint_path}\n"
        f"{top_path}\n{itp_path}\n0\n"
    )
    _run_sobtop_checked(mol2_path, commands, top_path, itp_path)


In [ ]:
# 创造单体的itp文件，力场参数基于量子化学的方法计算获取
'''
修改下列函数，函数的输入值为df
遍历Name列的每一行，记录元素为{name}，有
mol2_path = os.path.abspath(f"{name}" + '.mol2')
fchk_path = os.path.abspath(f"{name}" + '.fchk')
top_path = os.path.abspath(f"{name}" + '.top')
itp_path = os.path.abspath(f"{name}" + '.itp')
run_monomer_sobtop(mol2_path, fchk_path, top_path, itp_path)
'''
def create_monomer_itp_top(df):
    for name in df["Name"]:
        # mol2_path, chg_path, top_path, itp_path
        mol2_path = os.path.abspath(f"{name}" + '.mol2')
        fchk_path = os.path.abspath(f"{name}" + '.fchk')
        top_path = os.path.abspath(f"{name}" + '.top')
        itp_path = os.path.abspath(f"{name}" + '.itp')
        print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!{itp_path}!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        # 调用sobtop生成拓扑文件
        run_monomer_sobtop(mol2_path, fchk_path, top_path, itp_path)

In [ ]:
# 创建单体的itp文件
create_monomer_itp_top(df)

In [ ]:
'''
假设没有给出的函数已经被定义
给出一段函数，函数的输入值为df，遍历Name列，记为{name}，使用下列函数
atom_list = load_polymer_atom_list("{name}_atom_list.json")
然后在当前目录下查找并打开{name}.itp文件，
比较[ atoms ]中type列的长度是否与atom_list列表相等，如果相等则继续，如果不想等，则抛出错误，并且continue
然后将atom_list中的元素替换type，atom_list的索引顺序与type列的索引顺序对应。
然后修改[ bonds ]中的atom_i与atom_j，在没有修改前，这里的对应方法是是根据[ atoms ]的index列进行对应的，
现在我需要你将atom_i和atom_j列中的元素替换为[ atoms ]中type列的值。
对于[ angles ]也同理（需要替换atom_i，atom_j，atom_k列中的元素）
然后将新的itp文件保存在当前目录，记为fake_{name}.itp文件


已知itp文件中的格式如下所示：
[ atoms ]
;  Index   type   residue  resname   atom        cgnr     charge       mass
     1     c2         1      MOL     C             1   -0.10580000   12.010736
     2     c2         1      MOL     C             2   -0.10580000   12.010736
     3     ha         1      MOL     H             3    0.05290000    1.007941
     4     ha         1      MOL     H             4    0.05290000    1.007941
     5     ha         1      MOL     H             5    0.05290000    1.007941
     6     ha         1      MOL     H             6    0.05290000    1.007941
 
[ bonds ]
; atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)
    1       2         1        0.132703     5.373376E+05     ; C-C, mSeminario method
    1       5         1        0.108490     3.140224E+05     ; C-H, mSeminario method
    1       6         1        0.108487     3.140900E+05     ; C-H, mSeminario method
    2       3         1        0.108484     3.141551E+05     ; C-H, mSeminario method
    2       4         1        0.108490     3.140162E+05     ; C-H, mSeminario method
 
[ angles ]
; atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
    2       1       5         1       121.743      2.627051E+02     ; C-C-H, mSeminario method
    2       1       6         1       121.754      2.626640E+02     ; C-C-H, mSeminario method
    5       1       6         1       116.503      1.836433E+02     ; H-C-H, mSeminario method
    1       2       3         1       121.759      2.626248E+02     ; C-C-H, mSeminario method
    1       2       4         1       121.746      2.627095E+02     ; C-C-H, mSeminario method
    3       2       4         1       116.495      1.836274E+02     ; H-C-H, mSeminario method

'''
# -*- coding: utf-8 -*-
import os                             # 中文注释：用于文件存在性判断与路径操作
import re                             # 中文注释：用于匹配段落标题与处理空白
from typing import Dict, Tuple        # 中文注释：类型注解（可选）

# ========================= 内部小工具函数（仅在此函数作用域内使用） =========================
def _split_data_and_comment(line: str) -> Tuple[str, str]:
    """中文注释：将一行按第一个分号切分为“数据部分”和“注释部分（含分号）”"""
    pos = line.find(';')                          # 中文注释：查找第一个分号位置
    if pos == -1:                                 # 中文注释：若不存在行内注释
        return line.rstrip('\n'), ""              # 中文注释：返回数据部分与空注释
    return line[:pos].rstrip(), line[pos:].rstrip('\n')  # 中文注释：保留分号与注释内容

def _leading_ws(line: str) -> str:
    """中文注释：提取行首前导空白（用于尽量保持原文件缩进风格）"""
    m = re.match(r'^(\s*)', line)                 # 中文注释：匹配行首空白
    return m.group(1) if m else ""                # 中文注释：返回空白

def _is_section_header(line: str, name: str) -> bool:
    """中文注释：判断该行是否为形如 [ name ] 的段落标题（大小写不敏感、允许空格）"""
    pat = rf'^\s*\[\s*{re.escape(name)}\s*\]\s*$' # 中文注释：构造正则
    return re.match(pat, line, flags=re.IGNORECASE) is not None

def _find_section_bounds(lines):
    """中文注释：扫描文件行，记录各段起始下标（标题行的下标）；并给出段内容的范围 [start+1, end)"""
    headers = []                                   # 中文注释：收集 (lower_name, header_line_idx)
    for i, ln in enumerate(lines):                 # 中文注释：遍历每一行
        m = re.match(r'^\s*\[\s*([A-Za-z0-9_]+)\s*\]\s*.*$', ln)  # 放宽段名为含数字与下划线：匹配 形如[ name ] XXXX 的行
        if m:
            headers.append((m.group(1).lower(), i))          # 中文注释：记录段名与标题行行号
    bounds: Dict[str, Tuple[int, int]] = {}        # 中文注释：段名 -> (数据起始行, 数据结束行)
    for k, (sec_name, start_idx) in enumerate(headers):
        next_idx = headers[k + 1][1] if k + 1 < len(headers) else len(lines)  # 中文注释：下一段标题行或文件末尾
        bounds[sec_name] = (start_idx + 1, next_idx)  # 中文注释：数据范围不包含标题行本身
    return bounds

def _int_token(s: str) -> bool:
    """中文注释：判断一个token是否为整数（用于识别原子索引）"""
    return re.fullmatch(r'[+-]?\d+', s) is not None


def generate_fake_itp_from_atom_list(df) -> None:
    """
    中文功能说明：
        遍历 df 的 Name 列，读取 {name}_atom_list.json（通过已定义的 load_polymer_atom_list），
        打开当前目录下的 {name}.itp，比较 [ atoms ] 段 type 列条目数与 atom_list 长度是否一致；
        若一致则将 type 列替换为 atom_list，并将 [ bonds ] 与 [ angles ] 段中引用的
        索引（atom_i/atom_j/atom_k）改为对应的 type 字符串；保存为 fake_{name}.itp。
    """

    # ========================= 主流程 =========================
    if "Name" not in df.columns:                       # 中文注释：校验DataFrame是否包含Name列
        raise ValueError("输入 df 必须包含列：'Name'。")

    # 中文注释：按顺序遍历 Name 列
    for name in df["Name"]:
        name = str(name)                               # 中文注释：保证名称为字符串
        try:
            # 1) 加载 atom_list
            atom_list = load_polymer_atom_list(f"{name}_atom_list.json")  # 中文注释：按要求的固定调用方式
            if not isinstance(atom_list, list):        # 中文注释：类型健壮性校验
                raise TypeError(f"{name}: load_polymer_atom_list 未返回列表。")

            # 2) 打开 {name}.itp
            itp_path = os.path.join(".", f"{name}.itp")  # 中文注释：当前目录下的itp路径
            if not os.path.isfile(itp_path):             # 中文注释：若文件不存在则跳过
                print(f"[跳过] 未找到 {itp_path}")
                continue
            with open(itp_path, "r", encoding="utf-8") as f:
                lines = f.readlines()                    # 中文注释：一次性读取全部行

            # 3) 定位段范围
            bounds = _find_section_bounds(lines)         # 中文注释：获取各段数据范围
            if "atoms" not in bounds:
                raise ValueError(f"{name}: itp 文件中缺少 [ atoms ] 段。")
            atoms_s, atoms_e = bounds["atoms"]           # 中文注释：[ atoms ] 的数据行范围
            bonds_s, bonds_e = bounds.get("bonds", (None, None))   # 中文注释：允许无bonds段
            angles_s, angles_e = bounds.get("angles", (None, None))# 中文注释：允许无angles段

            # 4) 处理 [ atoms ]：统计数据行与替换 type 列
            idx_to_type_new: Dict[int, str] = {}         # 中文注释：Index -> 新 type 映射
            new_atom_lines = []                           # 中文注释：存放替换后的 [ atoms ] 数据行
            type_count = 0                                # 中文注释：计数：type列条目数（=数据行数）

            for li in range(atoms_s, atoms_e):
                raw = lines[li]                           # 中文注释：原行
                stripped = raw.strip()                    # 中文注释：剔除首尾空白以判断空行/注释
                if stripped == "" or stripped.startswith(";"):
                    new_atom_lines.append(raw)            # 中文注释：空行或注释行原样保留
                    continue

                data, cmt = _split_data_and_comment(raw)  # 中文注释：分离数据与注释
                toks = data.split()                       # 中文注释：以任意空白分割字段
                if len(toks) < 2:
                    # 中文注释：不够两个字段（缺少Index或Type），原样保留并警告
                    new_atom_lines.append(raw)
                    print(f"[警告] {name}: [ atoms ] 行格式异常，已跳过替换：{raw.rstrip()}")
                    continue

                # 中文注释：读取 Index（第一列）
                try:
                    idx_val = int(toks[0])                # 中文注释：将第一列转为整数索引
                except ValueError:
                    new_atom_lines.append(raw)            # 中文注释：若非整数索引，原样保留并警告
                    print(f"[警告] {name}: [ atoms ] 行首Index非整数，已跳过：{raw.rstrip()}")
                    continue

                # 中文注释：用 atom_list 的顺序替换第二列 type
                if type_count >= len(atom_list):
                    raise ValueError(f"{name}: atom_list 长度不足（需要≥{type_count+1}）。")
                new_type = str(atom_list[type_count])     # 中文注释：按顺序取新type
                toks[1] = new_type                        # 中文注释：替换type列
                type_count += 1                           # 中文注释：递增已替换计数

                idx_to_type_new[idx_val] = new_type       # 中文注释：记录Index->新type映射

                # 中文注释：重建该行（保留原前导空白与注释）
                lead = _leading_ws(raw)                   # 中文注释：原行前导空白
                rebuilt = lead + " ".join(toks)           # 中文注释：字段以单空格连接
                if cmt:                                   # 中文注释：附加原行内注释
                    if not rebuilt.endswith(" "):
                        rebuilt += "  "                   # 中文注释：在数据与注释间加两个空格以便阅读
                    rebuilt += cmt
                if not rebuilt.endswith("\n"):            # 中文注释：保证换行
                    rebuilt += "\n"
                new_atom_lines.append(rebuilt)            # 中文注释：写入新行

            # 中文注释：比较type列条目数与atom_list长度，不等则抛错并继续下一个name
            if type_count != len(atom_list):
                raise ValueError(f"{name}: [ atoms ] 段的 type 列条目数({type_count}) ≠ atom_list 长度({len(atom_list)})。")

            # 中文注释：将替换后的 [ atoms ] 数据写回
            lines[atoms_s:atoms_e] = new_atom_lines

            # 5) 处理 [ bonds ]：用新type替换 atom_i/atom_j（若存在）
            if bonds_s is not None and bonds_e is not None:
                new_bond_lines = []                       # 中文注释：存放替换后的 bonds 数据行
                for li in range(bonds_s, bonds_e):
                    raw = lines[li]
                    stripped = raw.strip()
                    if stripped == "" or stripped.startswith(";"):
                        new_bond_lines.append(raw)        # 中文注释：空行或注释行保持
                        continue
                    data, cmt = _split_data_and_comment(raw)
                    toks = data.split()
                    if len(toks) < 2 or (not _int_token(toks[0])) or (not _int_token(toks[1])):
                        # 中文注释：不满足前两列为整数索引的格式，原样保留
                        new_bond_lines.append(raw)
                        continue
                    # 中文注释：替换前两列索引为type
                    i_idx = int(toks[0])
                    j_idx = int(toks[1])
                    # 中文注释：若映射不存在则报错并保留原行
                    if i_idx not in idx_to_type_new or j_idx not in idx_to_type_new:
                        print(f"[警告] {name}: [ bonds ] 出现未在 [ atoms ] 中定义的索引（{i_idx} 或 {j_idx}），已保留原行。")
                        new_bond_lines.append(raw)
                        continue
                    toks[0] = idx_to_type_new[i_idx]
                    toks[1] = idx_to_type_new[j_idx]
                    lead = _leading_ws(raw)
                    rebuilt = lead + " ".join(toks)
                    if cmt:
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_bond_lines.append(rebuilt)
                lines[bonds_s:bonds_e] = new_bond_lines   # 中文注释：写回bonds段

            # 6) 处理 [ angles ]：用新type替换 atom_i/atom_j/atom_k（若存在）
            if angles_s is not None and angles_e is not None:
                new_angle_lines = []                      # 中文注释：存放替换后的 angles 数据行
                for li in range(angles_s, angles_e):
                    raw = lines[li]
                    stripped = raw.strip()
                    if stripped == "" or stripped.startswith(";"):
                        new_angle_lines.append(raw)       # 中文注释：空行或注释行保持
                        continue
                    data, cmt = _split_data_and_comment(raw)
                    toks = data.split()
                    if len(toks) < 3 or (not _int_token(toks[0])) or (not _int_token(toks[1])) or (not _int_token(toks[2])):
                        # 中文注释：不满足前三列为整数索引的格式，原样保留
                        new_angle_lines.append(raw)
                        continue
                    # 中文注释：替换前三列索引为type
                    i_idx = int(toks[0])
                    j_idx = int(toks[1])
                    k_idx = int(toks[2])
                    if (i_idx not in idx_to_type_new) or (j_idx not in idx_to_type_new) or (k_idx not in idx_to_type_new):
                        print(f"[警告] {name}: [ angles ] 出现未在 [ atoms ] 中定义的索引（{i_idx}/{j_idx}/{k_idx}），已保留原行。")
                        new_angle_lines.append(raw)
                        continue
                    toks[0] = idx_to_type_new[i_idx]
                    toks[1] = idx_to_type_new[j_idx]
                    toks[2] = idx_to_type_new[k_idx]
                    lead = _leading_ws(raw)
                    rebuilt = lead + " ".join(toks)
                    if cmt:
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_angle_lines.append(rebuilt)
                lines[angles_s:angles_e] = new_angle_lines # 中文注释：写回angles段

            # 7) 写出 fake_{name}.itp
            out_path = os.path.join(".", f"fake_{name}.itp")     # 中文注释：输出路径
            with open(out_path, "w", encoding="utf-8") as f:
                f.writelines(lines)                              # 中文注释：写回全部内容
            print(f"[完成] 已生成 {out_path}")

        except Exception as e:
            # 中文注释：满足“抛出错误，并且continue”的语义：打印错误信息后继续下一个name
            print(f"[错误] 处理 {name} 失败：{e}")
            continue

In [ ]:
# 创建使用原子名称{atom_type}_{name}代替索引与原子编号的单体fake itp文件
generate_fake_itp_from_atom_list(df)

In [ ]:
'''
请再给出一段函数，函数的输入值为df与polymer_node_list_path（默认值为"polymer_node_list.json"）：
读取df中的copolymer_name列内容（只需要第一个即可），记为{copolymer_name}
读取"polymer_node_list.json"（通过已定义的 load_polymer_atom_list）
打开当前目录下的{copolymer_name}.itp，比较 [ atoms ] 段 type 列条目数与 atom_list 长度是否一致；
若一致则将 type 列替换为 atom_list，并将 [ bonds ] 与 [ angles ] 段中引用的
索引（atom_i/atom_j/atom_k）改为对应的 type 字符串；保存为 fake_{copolymer_name}.itp。
注意内部小工具函数我已经在全局定义，不需要再重复给出
'''
# -*- coding: utf-8 -*-
def generate_fake_itp_for_copolymer(df, polymer_node_list_path: str = "polymer_node_list.json") -> None:
    """中文功能说明：
        1) 读取 df 中的第一条 copolymer_name，记为 {copolymer_name}；
        2) 通过已定义的 load_polymer_atom_list 读取 polymer_node_list_path（默认 "polymer_node_list.json"），得到 atom_list；
        3) 在当前目录打开 {copolymer_name}.itp，比较 [ atoms ] 段的 type 列条目数与 atom_list 长度是否一致；
        4) 若一致：用 atom_list 顺序替换 [ atoms ] 段第二列 type，并建立 Index->type 映射；
           再将 [ bonds ] 的 atom_i/atom_j、[ angles ] 的 atom_i/atom_j/atom_k 中的“索引数字”
           替换为对应的“type 字符串”；
        5) 保存为 fake_{copolymer_name}.itp。
    """
    import os  # 中文注释：用于文件路径与存在性判断
    # ========================= 0) 基本输入校验 =========================
    if "copolymer_name" not in df.columns:  # 中文注释：必须存在 copolymer_name 列
        raise ValueError("输入 df 必须包含列：'copolymer_name'。")
    if len(df) == 0:  # 中文注释：df 不能为空
        raise ValueError("输入 df 为空，无法获取 copolymer_name。")

    # 中文注释：读取第一条 copolymer_name，转换为字符串并做基本校验
    copolymer_name = str(df["copolymer_name"].iloc[0])  # 中文注释：只取第一条记录
    if (copolymer_name is None) or (copolymer_name.strip() == "") or (copolymer_name.lower() == "nan"):
        raise ValueError("第一条 copolymer_name 为空或无效。")

    # ========================= 1) 读取 atom_list =========================
    atom_list = load_polymer_atom_list(polymer_node_list_path)  # 中文注释：从 polymer_node_list_path 读取节点列表

    # 将形如“{name}_{i}”的字符串统一裁剪为“{name}”；仅当末尾为“_纯数字”时才去掉
    '''
    由于atom_list = load_polymer_atom_list(polymer_node_list_path)的atom_list中的元素形如：“C0_test_C_C_20250821_1”，其结构为“{name}_{i}”，即
    name为C0_test_C_C_20250821，也有可能是其他任何输入值，而{i}在这里为1，
    现在我希望你在原来代码的基础上多补充代码，作用为：
    将结构为“{name}_{i}”的元素全部转化为“{name}”，你不需要给出全部代码，只需要告诉我应该在哪里修改以及具体增补代码即可。
    '''
    _suffix_num_pat = re.compile(r"_(\d+)$")  # 中文注释：匹配行尾的“_数字”
    atom_list = [_suffix_num_pat.sub("", str(x)) for x in atom_list]  # 中文注释：批量去掉尾部编号

    if not isinstance(atom_list, list):  # 中文注释：必须为列表
        raise TypeError(f"{polymer_node_list_path} 未返回列表类型。")

    # ========================= 2) 读取 ITP 文件全文 =========================
    itp_path = os.path.join(".", f"{copolymer_name}.itp")  # 中文注释：当前目录下的 ITP 文件路径
    if not os.path.isfile(itp_path):  # 中文注释：文件必须存在
        raise FileNotFoundError(f"未找到 ITP 文件：{itp_path}")
    with open(itp_path, "r", encoding="utf-8") as f:  # 中文注释：一次性读入所有行
        lines = f.readlines()

    # ========================= 3) 定位段落范围 =========================
    bounds = _find_section_bounds(lines)  # 中文注释：获取 [atoms]/[bonds]/[angles] 的数据起止行号（不含标题行）
    if "atoms" not in bounds:  # 中文注释：必须存在 [ atoms ] 段
        raise ValueError(f"{copolymer_name}: itp 文件缺少 [ atoms ] 段。")
    atoms_s, atoms_e = bounds["atoms"]  # 中文注释：[ atoms ] 数据行范围（左闭右开）
    bonds_range = bounds.get("bonds", (None, None))  # 中文注释：可选 [ bonds ] 段
    angles_range = bounds.get("angles", (None, None))  # 中文注释：可选 [ angles ] 段
    bonds_s, bonds_e = bonds_range  # 中文注释：若不存在则为 (None, None)
    angles_s, angles_e = angles_range  # 中文注释：若不存在则为 (None, None)

    # ========================= 4) 统计 [ atoms ] 中的“type条目数” =========================
    type_entries = 0  # 中文注释：统计可替换的“数据行”数量（即将被替换的 type 数量）
    for li in range(atoms_s, atoms_e):  # 中文注释：遍历 [ atoms ] 数据区
        raw = lines[li]  # 中文注释：原始行
        stripped = raw.strip()  # 中文注释：去掉前后空白以判断行类型
        if stripped == "" or stripped.startswith(";"):  # 中文注释：空行或注释行不计入
            continue
        data, _cmt = _split_data_and_comment(raw)  # 中文注释：分离“数据部分”和“行内注释”
        toks = data.split()  # 中文注释：按空白拆分字段
        if len(toks) >= 2:  # 中文注释：至少应有 Index 与 Type 两列
            # 中文注释：要求首列为整数索引，才视为有效“数据行”
            if _int_token(toks[0]):
                type_entries += 1  # 中文注释：计数有效数据行

    # 中文注释：与 atom_list 长度进行严格比较
    if type_entries != len(atom_list):  # 中文注释：长度不一致则抛出错误
        raise ValueError(
            f"{copolymer_name}: [ atoms ] 段的 type 条目数({type_entries}) ≠ atom_list 长度({len(atom_list)})。"
        )

    # ========================= 5) 替换 [ atoms ] 的 type，并建立 Index->type 映射 =========================
    idx_to_type_new = {}  # 中文注释：Index(整数) -> 新 type(字符串) 的映射表
    new_atom_lines = []  # 中文注释：存放替换后的 [ atoms ] 数据行
    take = 0  # 中文注释：atom_list 的读取进度指针
    for li in range(atoms_s, atoms_e):  # 中文注释：再次遍历 [ atoms ] 数据区进行实际替换
        raw = lines[li]  # 中文注释：原行
        stripped = raw.strip()  # 中文注释：去空白判定
        if stripped == "" or stripped.startswith(";"):  # 中文注释：空行/注释行原样保留
            new_atom_lines.append(raw)  # 中文注释：直接加入
            continue
        data, cmt = _split_data_and_comment(raw)  # 中文注释：数据与注释分离
        toks = data.split()  # 中文注释：拆分字段
        if len(toks) < 2 or (not _int_token(toks[0])):  # 中文注释：若格式异常则原样保留并给提示
            new_atom_lines.append(raw)  # 中文注释：不替换异常行
            print(f"[警告] {copolymer_name}: [ atoms ] 行格式异常，已跳过：{raw.rstrip()}")
            continue
        idx_val = int(toks[0])  # 中文注释：读取第一列Index
        new_type = str(atom_list[take])  # 中文注释：按顺序取出对应的新类型名
        toks[1] = new_type  # 中文注释：替换第二列 type
        idx_to_type_new[idx_val] = new_type  # 中文注释：记录 Index->type 映射
        take += 1  # 中文注释：移动指针
        lead = _leading_ws(raw)  # 中文注释：保持原行的前导空白风格
        rebuilt = lead + " ".join(toks)  # 中文注释：以单空格重建字段
        if cmt:  # 中文注释：若存在行内注释则拼接
            if not rebuilt.endswith(" "):
                rebuilt += "  "  # 中文注释：数据与注释之间留两个空格以增强可读性
            rebuilt += cmt  # 中文注释：附加原注释（含分号）
        if not rebuilt.endswith("\n"):  # 中文注释：保证行末换行
            rebuilt += "\n"
        new_atom_lines.append(rebuilt)  # 中文注释：加入新行

    # 中文注释：将替换后的 [ atoms ] 写回整体行列表
    lines[atoms_s:atoms_e] = new_atom_lines  # 中文注释：切片赋值以覆盖原数据区

    # ========================= 6) 替换 [ bonds ] 的 atom_i/atom_j（若存在） =========================
    if (bonds_s is not None) and (bonds_e is not None):  # 中文注释：仅当存在 [ bonds ] 段时处理
        new_bond_lines = []  # 中文注释：存放替换后的 bonds 行
        for li in range(bonds_s, bonds_e):  # 中文注释：遍历 bonds 数据区
            raw = lines[li]  # 中文注释：原行
            stripped = raw.strip()  # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"):  # 中文注释：空行/注释行保持不变
                new_bond_lines.append(raw)  # 中文注释：写回原行
                continue
            data, cmt = _split_data_and_comment(raw)  # 中文注释：分离数据与注释
            toks = data.split()  # 中文注释：拆分字段
            # 中文注释：前两列必须为整数索引，才进行替换
            if len(toks) >= 2 and _int_token(toks[0]) and _int_token(toks[1]):
                i_idx = int(toks[0])  # 中文注释：atom_i 原索引
                j_idx = int(toks[1])  # 中文注释：atom_j 原索引
                if (i_idx in idx_to_type_new) and (j_idx in idx_to_type_new):  # 中文注释：只有映射存在才替换
                    toks[0] = idx_to_type_new[i_idx]  # 中文注释：替换为新type
                    toks[1] = idx_to_type_new[j_idx]  # 中文注释：替换为新type
                    lead = _leading_ws(raw)  # 中文注释：保留前导空白
                    rebuilt = lead + " ".join(toks)  # 中文注释：重建行
                    if cmt:  # 中文注释：拼接原注释
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_bond_lines.append(rebuilt)  # 中文注释：加入替换后的行
                else:
                    print(f"[警告] {copolymer_name}: [ bonds ] 存在未映射索引({i_idx} 或 {j_idx})，已保留原行。")
                    new_bond_lines.append(raw)  # 中文注释：无映射则保留原行
            else:
                new_bond_lines.append(raw)  # 中文注释：格式不满足替换条件则保留原行
        lines[bonds_s:bonds_e] = new_bond_lines  # 中文注释：写回 bonds 段

    # ========================= 7) 替换 [ angles ] 的 atom_i/atom_j/atom_k（若存在） =========================
    if (angles_s is not None) and (angles_e is not None):  # 中文注释：仅当存在 [ angles ] 段时处理
        new_angle_lines = []  # 中文注释：存放替换后的 angles 行
        for li in range(angles_s, angles_e):  # 中文注释：遍历 angles 数据区
            raw = lines[li]  # 中文注释：原行
            stripped = raw.strip()  # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"):  # 中文注释：空行/注释行保持
                new_angle_lines.append(raw)  # 中文注释：写回原行
                continue
            data, cmt = _split_data_and_comment(raw)  # 中文注释：分离数据与注释
            toks = data.split()  # 中文注释：拆分字段
            # 中文注释：前三列必须为整数索引，才进行替换
            if len(toks) >= 3 and _int_token(toks[0]) and _int_token(toks[1]) and _int_token(toks[2]):
                i_idx = int(toks[0])  # 中文注释：atom_i 原索引
                j_idx = int(toks[1])  # 中文注释：atom_j 原索引
                k_idx = int(toks[2])  # 中文注释：atom_k 原索引
                if (i_idx in idx_to_type_new) and (j_idx in idx_to_type_new) and (k_idx in idx_to_type_new):  # 中文注释：映射需齐全
                    toks[0] = idx_to_type_new[i_idx]  # 中文注释：替换为新type
                    toks[1] = idx_to_type_new[j_idx]  # 中文注释：替换为新type
                    toks[2] = idx_to_type_new[k_idx]  # 中文注释：替换为新type
                    lead = _leading_ws(raw)  # 中文注释：保留前导空白
                    rebuilt = lead + " ".join(toks)  # 中文注释：重建行
                    if cmt:  # 中文注释：拼接原注释
                        if not rebuilt.endswith(" "):
                            rebuilt += "  "
                        rebuilt += cmt
                    if not rebuilt.endswith("\n"):
                        rebuilt += "\n"
                    new_angle_lines.append(rebuilt)  # 中文注释：加入替换后的行
                else:
                    print(f"[警告] {copolymer_name}: [ angles ] 存在未映射索引({i_idx}/{j_idx}/{k_idx})，已保留原行。")
                    new_angle_lines.append(raw)  # 中文注释：无映射则保留原行
            else:
                new_angle_lines.append(raw)  # 中文注释：格式不满足替换条件则保留原行
        lines[angles_s:angles_e] = new_angle_lines  # 中文注释：写回 angles 段

    # ========================= 8) 写出 fake_{copolymer_name}.itp =========================
    out_path = os.path.join(".", f"fake_{copolymer_name}.itp")  # 中文注释：输出路径
    with open(out_path, "w", encoding="utf-8") as f:  # 中文注释：写回全部行
        f.writelines(lines)  # 中文注释：保存修改后的 itp 内容
    print(f"[完成] 已生成 {out_path}")  # 中文注释：终端提示

In [ ]:
# 创建使用原子名称{atom_type}_{name}代替索引与原子编号的copolymer的fake itp文件
generate_fake_itp_for_copolymer(df, polymer_node_list_path="polymer_node_list.json")


In [ ]:
'''
给出一段函数，函数的输入值为df
创建2个文件，一个为monomer_bonds，另一个为monomer_angles
遍历Name列，打开当前目录下的fake_{name}.itp文件
对于[ bonds ]，将所有[bond]中的内容追加到monomer_bonds中
对于[ angles ]，将所有[angles]中的内容追加到monomer_angles中
直至完成遍历。并且保存在当前目录下并且返回（格式自行选择）

然后再给出一段函数，函数的输入值为fake_copolymer_itp_path
根据fake_copolymer_itp_path打开该itp文件，找到[bond]，遍历[bond]中的每一行
由于[bond]形如atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)：
; atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)
C0_test_C_C_20250821 C1_test_C_C_20250821 1 0.133430 4.764739E+05  ; C-C, prebuilt c2-c2
所以你需要从monomer_bonds中找到atom_i  atom_j与fake_copolymer_itp_path的atom_i  atom_j配对的行，以替换掉fake_copolymer_itp_path中对应的行，
对于找不到对应atom_i  atom_j的情况，则跳过。直至替换完所有[bond]。

然后找到fake_copolymer_itp_path的[ angles ]，其形如：atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
; atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
    C1_test_C_C_20250821 C0_test_C_C_20250821 H4_test_C_C_20250821 1 121.743 2.627051E+02  ; C-C-H, mSeminario method
所以你需要从monomer_angles中找到atom_i  atom_j  atom_k与fake_copolymer_itp_path的atom_i  atom_j  atom_k配对的行，以替换掉fake_copolymer_itp_path中对应的行，
对于找不到对应atom_i  atom_j  atom_k的情况，则跳过。直至替换完所有[bond]。

将完成上诉步骤的文件保存在当前目录下，记为replace_{fake_copolymer_itp_path}（注意fake_copolymer_itp_path这里只需要提取文件名和后缀名，不需要路径）

然后再给出一段函数，函数的输入值为copolymer_itp_path以及replace_fake_copolymer_itp_path，
找到copolymer_itp_path与replace_fake_copolymer_itp_path的
[ bonds ]
; atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)

然后使用replace_fake_copolymer_itp_path的后三列数据functype      r0 (nm)    k (kJ/mol/nm^2)
替换掉copolymer_itp_path的后三列数据functype      r0 (nm)    k (kJ/mol/nm^2)
然后找到
[ angles ]
; atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
然后使用replace_fake_copolymer_itp_path的后三列数据functype    a0 (Deg.)  k (kJ/mol/rad^2)
替换掉copolymer_itp_path的后三列数据functype    a0 (Deg.)  k (kJ/mol/rad^2)
其他部分保持原样，将新的copolymer_itp_path保存为calculated_{copolymer_itp_path}（注意copolymer_itp_path这里只需要提取文件名和后缀名，不需要路径）

'''

In [ ]:
# -*- coding: utf-8 -*-
import os  # 中文注释：处理路径与文件操作
'''
这段函数似乎还存在错误：
虽然bonds能够成功提取，但对于[ angles ]，提取存在错误：
具体而言，[ angles ]的提取区间，应该是在[ angles ]行与[ dihedrals ] ; propers行（注意识别到dihedrals即可）之间，请修改下列函数，重新给出结果。
'''
def build_monomer_bonds_angles(df):
    """
    中文功能说明：
        遍历 df['Name']，打开当前目录下的 fake_{name}.itp，
        将其中 [ bonds ] 段的所有数据行（不含空行和注释行）追加到 monomer_bonds.txt，
        将 [ angles ] 段的所有数据行（不含空行和注释行）追加到 monomer_angles.txt。
        最终在当前目录生成两个文件并返回其路径。
    """
    # 中文注释：校验 df 中必须含 Name 列
    if "Name" not in df.columns:
        raise ValueError("输入 df 必须包含列：'Name'。")

    # 中文注释：用于收集所有单体的 bond 与 angle 数据行（仅数据部分 + 保留原行内注释）
    all_bonds = []   # 中文注释：类型为 List[str]
    all_angles = []  # 中文注释：类型为 List[str]

    # 中文注释：遍历 df['Name'] 列
    for name in df["Name"]:
        # 中文注释：构造 fake_{name}.itp 的路径（当前目录）
        itp_path = os.path.join(".", f"fake_{name}.itp")
        # 中文注释：若文件不存在则跳过并提示
        if not os.path.isfile(itp_path):
            print(f"[跳过] 未找到 {itp_path}")
            continue

        # 中文注释：读入整文件行
        with open(itp_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        # 中文注释：定位各段范围（使用已定义的 _find_section_bounds）
        bounds = _find_section_bounds(lines)
        bonds_s, bonds_e = bounds.get("bonds", (None, None))
        angles_s, angles_e = bounds.get("angles", (None, None))

        # 中文注释：抽取 [ bonds ] 数据行
        if bonds_s is not None and bonds_e is not None:
            for li in range(bonds_s, bonds_e):
                raw = lines[li]                 # 中文注释：原始行
                stripped = raw.strip()          # 中文注释：去除首尾空白以判断类型
                if stripped == "" or stripped.startswith(";"):
                    continue                    # 中文注释：跳过空行与注释行
                data, cmt = _split_data_and_comment(raw)  # 中文注释：拆分数据与注释
                # 中文注释：收集为“数据 + 两空格 + 注释”（若无注释则仅数据）
                all_bonds.append((data.strip() + ("" if not cmt else "  " + cmt)).rstrip())

        # 中文注释：抽取 [ angles ] 数据行
        if angles_s is not None and angles_e is not None:
            for li in range(angles_s, angles_e):
                raw = lines[li]                 # 中文注释：原始行
                stripped = raw.strip()          # 中文注释：去除首尾空白以判断类型
                if stripped == "" or stripped.startswith(";"):
                    continue                    # 中文注释：跳过空行与注释行
                data, cmt = _split_data_and_comment(raw)  # 中文注释：拆分数据与注释
                # 中文注释：收集为“数据 + 两空格 + 注释”（若无注释则仅数据）
                all_angles.append((data.strip() + ("" if not cmt else "  " + cmt)).rstrip())

    # 中文注释：写出到当前目录下的文本文件
    bonds_path = os.path.join(".", "monomer_bonds.txt")     # 中文注释：输出文件路径
    angles_path = os.path.join(".", "monomer_angles.txt")   # 中文注释：输出文件路径
    with open(bonds_path, "w", encoding="utf-8") as fb:
        for line in all_bonds:
            fb.write(line + "\n")
    with open(angles_path, "w", encoding="utf-8") as fa:
        for line in all_angles:
            fa.write(line + "\n")

    # 中文注释：返回两个文件路径
    return bonds_path, angles_path

In [ ]:
# 创建单体力常数单体bonds与angles的映射关系
bonds_path, angles_path = build_monomer_bonds_angles(df)

In [ ]:
# -*- coding: utf-8 -*-
def replace_fake_copolymer_from_monomers(fake_copolymer_itp_path):
    """
    中文功能说明：
        使用当前目录下的 monomer_bonds.txt 与 monomer_angles.txt，
        将 fake_copolymer_itp_path 文件中 [ bonds ] 与 [ angles ] 段内的匹配行替换为
        单体库中对应的整行文本（数据与注释），匹配键分别为：
            bonds:  (atom_i, atom_j)  —— 同时支持 (atom_j, atom_i) 的反向匹配
            angles: (atom_i, atom_j, atom_k)
        其他行保持不变，最终输出为 ./replace_{basename(fake_copolymer_itp_path)}。
    """
    import os  # 中文注释：路径与文件操作

    # 中文注释：准备输入与输出路径
    base_name = os.path.basename(fake_copolymer_itp_path)         # 中文注释：仅取文件名与后缀
    out_path = os.path.join(".", f"replace_{base_name}")          # 中文注释：输出文件路径

    # 中文注释：读取 monomer_bonds / monomer_angles 两个库文件（必须存在）
    bonds_lib = os.path.join(".", "monomer_bonds.txt")
    angles_lib = os.path.join(".", "monomer_angles.txt")
    if not os.path.isfile(bonds_lib) or not os.path.isfile(angles_lib):
        raise FileNotFoundError("未找到 monomer_bonds.txt 或 monomer_angles.txt，请先运行构建函数。")

    # 中文注释：构建 bonds 映射表： (atom_i, atom_j) -> 完整行（数据+注释）
    bonds_map = {}  # 中文注释：Dict[Tuple[str, str], str]
    with open(bonds_lib, "r", encoding="utf-8") as fb:
        for raw in fb:
            line = raw.strip()                     # 中文注释：去首尾空白
            if line == "" or line.startswith(";"):  # 中文注释：防御性跳过
                continue
            data, cmt = _split_data_and_comment(line)  # 中文注释：分离数据与注释
            toks = data.split()                        # 中文注释：拆分字段
            if len(toks) < 3:                          # 中文注释：至少应有 atom_i atom_j functype
                continue
            k = (toks[0], toks[1])                     # 中文注释：键为(第一列, 第二列)
            full_line = data + ("" if not cmt else "  " + cmt)  # 中文注释：重组为“数据 + 注释”
            bonds_map[k] = full_line                   # 中文注释：正向键
            bonds_map[(k[1], k[0])] = full_line       # 中文注释：反向键（处理无序键）

    # 中文注释：构建 angles 映射表： (atom_i, atom_j, atom_k) -> 完整行（数据+注释）
    angles_map = {}  # 中文注释：Dict[Tuple[str, str, str], str]
    with open(angles_lib, "r", encoding="utf-8") as fa:
        for raw in fa:
            line = raw.strip()                     # 中文注释：去首尾空白
            if line == "" or line.startswith(";"):  # 中文注释：防御性跳过
                continue
            data, cmt = _split_data_and_comment(line)  # 中文注释：分离数据与注释
            toks = data.split()                        # 中文注释：拆分字段
            if len(toks) < 4:                          # 中文注释：至少应有 atom_i atom_j atom_k functype
                continue
            k = (toks[0], toks[1], toks[2])            # 中文注释：键为前三列
            full_line = data + ("" if not cmt else "  " + cmt)  # 中文注释：重组为“数据 + 注释”
            angles_map[k] = full_line                  # 中文注释：登记映射

    # 中文注释：读取目标 fake_copolymer_itp
    with open(fake_copolymer_itp_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # 中文注释：定位段范围
    bounds = _find_section_bounds(lines)
    bonds_s, bonds_e = bounds.get("bonds", (None, None))
    angles_s, angles_e = bounds.get("angles", (None, None))

    # 中文注释：处理 [ bonds ] 段：若匹配到 (i,j) 则以库中整行替换
    if bonds_s is not None and bonds_e is not None:
        new_bond_lines = []                                # 中文注释：存放替换后的行
        for li in range(bonds_s, bonds_e):
            raw = lines[li]                                # 中文注释：原始行
            stripped = raw.strip()                         # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"): # 中文注释：空行/注释行保持不变
                new_bond_lines.append(raw)
                continue
            data, cmt_old = _split_data_and_comment(raw)   # 中文注释：分离原数据与原注释
            toks = data.split()                            # 中文注释：拆分字段
            if len(toks) < 2:                              # 中文注释：不足以定位键则保留原行
                new_bond_lines.append(raw)
                continue
            key = (toks[0], toks[1])                       # 中文注释：键=前两列
            if key in bonds_map:                           # 中文注释：若命中映射
                repl = bonds_map[key]                      # 中文注释：替换行（数据+注释）
                lead = _leading_ws(raw)                    # 中文注释：保留原行的前导空白
                new_line = lead + repl                     # 中文注释：拼接为新行
                if not new_line.endswith("\n"):
                    new_line += "\n"
                new_bond_lines.append(new_line)            # 中文注释：写入替换结果
            else:
                new_bond_lines.append(raw)                 # 中文注释：未命中则保持原样
        lines[bonds_s:bonds_e] = new_bond_lines            # 中文注释：写回切片

    # 中文注释：处理 [ angles ] 段：若匹配到 (i,j,k) 则以库中整行替换
    if angles_s is not None and angles_e is not None:
        new_angle_lines = []                               # 中文注释：存放替换后的行
        for li in range(angles_s, angles_e):
            raw = lines[li]                                # 中文注释：原始行
            stripped = raw.strip()                         # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"): # 中文注释：空行/注释行保持不变
                new_angle_lines.append(raw)
                continue
            data, cmt_old = _split_data_and_comment(raw)   # 中文注释：分离原数据与原注释
            toks = data.split()                            # 中文注释：拆分字段
            if len(toks) < 3:                              # 中文注释：不足以定位键则保留原行
                new_angle_lines.append(raw)
                continue
            key = (toks[0], toks[1], toks[2])              # 中文注释：键=前三列
            if key in angles_map:                          # 中文注释：若命中映射
                repl = angles_map[key]                     # 中文注释：替换行（数据+注释）
                lead = _leading_ws(raw)                    # 中文注释：保留原行的前导空白
                new_line = lead + repl                     # 中文注释：拼接为新行
                if not new_line.endswith("\n"):
                    new_line += "\n"
                new_angle_lines.append(new_line)           # 中文注释：写入替换结果
            else:
                new_angle_lines.append(raw)                # 中文注释：未命中则保持原样
        lines[angles_s:angles_e] = new_angle_lines         # 中文注释：写回切片

    # 中文注释：写出到当前目录 replace_{basename(fake_copolymer_itp_path)}
    with open(out_path, "w", encoding="utf-8") as fo:
        fo.writelines(lines)

    # 中文注释：返回输出路径
    return out_path

In [ ]:
copolymer_name = df["copolymer_name"][0] # 从df中提取聚合物的名称
out_path = replace_fake_copolymer_from_monomers(f"fake_{copolymer_name}.itp")

In [ ]:
# -*- coding: utf-8 -*-
'''
你需要修改函数apply_calculated_params_to_copolymer(copolymer_itp_path, replace_fake_copolymer_itp_path)
目前似乎没有完成替换的任务，我再次重申要求：
然后再给出一段函数，函数的输入值为copolymer_itp_path以及replace_fake_copolymer_itp_path，
找到copolymer_itp_path与replace_fake_copolymer_itp_path的
其中，copolymer_itp_path形如
[ bonds ]
; atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)
    1       2         1        0.133430     4.764739E+05     ; C-C, prebuilt c2-c2

其中，replace_fake_copolymer_itp_path形如
[ bonds ]
; atom_i  atom_j  functype      r0 (nm)    k (kJ/mol/nm^2)
    C0_test_C_C_20250821 C1_test_C_C_20250821 1 0.132703 5.373376E+05  ; C-C, mSeminario method

然后使用replace_fake_copolymer_itp_path的后4列数据functype      r0 (nm)    k (kJ/mol/nm^2)以及注释（1 0.132703 5.373376E+05  ; C-C, mSeminario method）
替换掉copolymer_itp_path的后4列数据functype      r0 (nm)    k (kJ/mol/nm^2)以及注释（1 0.132703 5.373376E+05  ; C-C, mSeminario method）
然后找到
其中，copolymer_itp_path形如
[ angles ]
; atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
    2       1       4         1       120.430      4.175632E+02     ; C-C-H, prebuilt c2-c2-ha

其中，replace_fake_copolymer_itp_path形如
[ angles ]
; atom_i  atom_j  atom_k  functype    a0 (Deg.)  k (kJ/mol/rad^2)
    C1_test_C_C_20250821 C0_test_C_C_20250821 H4_test_C_C_20250821 1 121.743 2.627051E+02  ; C-C-H, mSeminario method

然后使用replace_fake_copolymer_itp_path的后4列数据functype    a0 (Deg.)  k (kJ/mol/rad^2)以及注释（1       120.430      4.175632E+02     ; C-C-H, prebuilt c2-c2-ha）
替换掉copolymer_itp_path的后三列数据functype    a0 (Deg.)  k (kJ/mol/rad^2)以及注释（1 121.743 2.627051E+02  ; C-C-H, mSeminario method）
其他部分保持原样，将新的copolymer_itp_path保存为calculated_{copolymer_itp_path}（注意copolymer_itp_path这里只需要提取文件名和后缀名，不需要路径）
'''
# -*- coding: utf-8 -*-
def apply_calculated_params_to_copolymer(copolymer_itp_path, replace_fake_copolymer_itp_path):
    """
    中文功能说明：
        使用 replace_fake_copolymer_itp_path 中的参数，更新 copolymer_itp_path：
        - [bonds]：用 replace_fake 的“后四列（functype r0 k）+ 注释”替换 copolymer 的后四列+注释；
        - [angles]：用 replace_fake 的“后四列（functype a0 k）+ 注释”替换 copolymer 的后四列+注释；
        （注意：这里“后四列”指代 3 个数据列 + 行内注释。）
        保留 copolymer 原行的前导空白与前两列（或前三列）索引不变。
        输出到 ./calculated_{basename(copolymer_itp_path)} 并返回其路径。
    """
    import os  # 中文注释：路径与文件操作

    # =================== 1) 读取 replace_fake 文件，建立映射 ===================
    with open(replace_fake_copolymer_itp_path, "r", encoding="utf-8") as fr:
        r_lines = fr.readlines()
    r_bounds = _find_section_bounds(r_lines)

    # 中文注释：构建 Index->type 映射（来自 replace_fake 的 [atoms] 段）
    idx_to_type = {}  # 中文注释：Dict[int, str]
    if "atoms" not in r_bounds:
        raise ValueError("replace_fake_copolymer_itp_path 缺少 [ atoms ] 段，无法建立 Index→type 映射。")
    r_atoms_s, r_atoms_e = r_bounds["atoms"]
    for li in range(r_atoms_s, r_atoms_e):
        raw = r_lines[li]                                  # 中文注释：原行
        stripped = raw.strip()                             # 中文注释：去除首尾空白
        if stripped == "" or stripped.startswith(";"):     # 中文注释：跳过空行与注释
            continue
        data, _ = _split_data_and_comment(raw)             # 中文注释：分离数据与注释
        toks = data.split()                                # 中文注释：分词
        if len(toks) >= 2 and _int_token(toks[0]):         # 中文注释：至少包含 Index 与 type
            idx_to_type[int(toks[0])] = toks[1]            # 中文注释：登记索引到type

    # 中文注释：从 replace_fake 的 [bonds]/[angles] 建立“键 → 后四列+注释”的映射
    r_bonds_s, r_bonds_e = r_bounds.get("bonds", (None, None))
    r_angles_s, r_angles_e = r_bounds.get("angles", (None, None))

    bonds_tail_map = {}  # 中文注释：Dict[Tuple[str, str], str]  # 值为“functype r0 k + 注释”
    if r_bonds_s is not None and r_bonds_e is not None:
        for li in range(r_bonds_s, r_bonds_e):
            raw = r_lines[li]
            stripped = raw.strip()
            if stripped == "" or stripped.startswith(";"):
                continue
            data, cmt = _split_data_and_comment(raw)
            toks = data.split()
            if len(toks) < 3:  # 中文注释：至少 atom_i atom_j functype
                continue
            key = (toks[0], toks[1])                    # 中文注释：类型键（非数字）
            tail_tokens = toks[2:]                      # 中文注释：从 functype 起的所有数据列
            tail = " ".join(tail_tokens)                # 中文注释：拼接后三列数据
            if cmt:
                tail += "  " + cmt                      # 中文注释：附加行内注释（含分号）
            bonds_tail_map[key] = tail                  # 中文注释：正向键
            bonds_tail_map[(key[1], key[0])] = tail     # 中文注释：反向键（bonds 对称）

    angles_tail_map = {}  # 中文注释：Dict[Tuple[str, str, str], str]  # 值为“functype a0 k + 注释”
    if r_angles_s is not None and r_angles_e is not None:
        for li in range(r_angles_s, r_angles_e):
            raw = r_lines[li]
            stripped = raw.strip()
            if stripped == "" or stripped.startswith(";"):
                continue
            data, cmt = _split_data_and_comment(raw)
            toks = data.split()
            if len(toks) < 4:  # 中文注释：至少 atom_i atom_j atom_k functype
                continue
            key = (toks[0], toks[1], toks[2])           # 中文注释：类型三元组键
            tail_tokens = toks[3:]                      # 中文注释：从 functype 起的所有数据列
            tail = " ".join(tail_tokens)                # 中文注释：拼接后三列数据
            if cmt:
                tail += "  " + cmt                      # 中文注释：附加行内注释（含分号）
            angles_tail_map[key] = tail                 # 中文注释：登记映射（角通常有方向性，不做反向）

    # =================== 2) 读取 copolymer 文件并按映射替换 ===================
    with open(copolymer_itp_path, "r", encoding="utf-8") as fc:
        c_lines = fc.readlines()
    c_bounds = _find_section_bounds(c_lines)
    c_bonds_s, c_bonds_e = c_bounds.get("bonds", (None, None))
    c_angles_s, c_angles_e = c_bounds.get("angles", (None, None))

    # ---------- 替换 [bonds] 的“后四列（functype r0 k）+ 注释” ----------
    if c_bonds_s is not None and c_bonds_e is not None:
        new_c_bonds = []                                   # 中文注释：存放替换后的行
        for li in range(c_bonds_s, c_bonds_e):
            raw = c_lines[li]                              # 中文注释：原行
            stripped = raw.strip()                         # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"): # 中文注释：空行或注释原样保留
                new_c_bonds.append(raw)
                continue
            data, _orig_cmt = _split_data_and_comment(raw) # 中文注释：拆分数据与注释（原注释将被新注释覆盖）
            toks = data.split()
            if len(toks) < 2 or (not _int_token(toks[0])) or (not _int_token(toks[1])):
                new_c_bonds.append(raw)                    # 中文注释：前两列非整数索引则跳过
                continue
            i_idx, j_idx = int(toks[0]), int(toks[1])      # 中文注释：读取数字索引
            ti, tj = idx_to_type.get(i_idx), idx_to_type.get(j_idx)  # 中文注释：映射为类型键
            if ti is None or tj is None:
                new_c_bonds.append(raw)                    # 中文注释：若无映射则保持原样
                continue
            tail = bonds_tail_map.get((ti, tj))            # 中文注释：获取“后四列+注释”
            if tail is None:
                new_c_bonds.append(raw)                    # 中文注释：找不到匹配则保持原样
                continue
            # 中文注释：重建行：保留前导空白与前两列索引，用新的尾部（含注释）
            lead = _leading_ws(raw)
            new_line = f"{lead}{toks[0]} {toks[1]} {tail}"
            if not new_line.endswith("\n"):
                new_line += "\n"
            new_c_bonds.append(new_line)
        c_lines[c_bonds_s:c_bonds_e] = new_c_bonds        # 中文注释：切片写回

    # ---------- 替换 [angles] 的“后四列（functype a0 k）+ 注释” ----------
    if c_angles_s is not None and c_angles_e is not None:
        new_c_angles = []                                  # 中文注释：存放替换后的行
        for li in range(c_angles_s, c_angles_e):
            raw = c_lines[li]                              # 中文注释：原行
            stripped = raw.strip()                         # 中文注释：去空白
            if stripped == "" or stripped.startswith(";"): # 中文注释：空行或注释原样保留
                new_c_angles.append(raw)
                continue
            data, _orig_cmt = _split_data_and_comment(raw) # 中文注释：拆分数据与注释（原注释将被新注释覆盖）
            toks = data.split()
            # 中文注释：前三列必须为整数索引
            if len(toks) < 3 or (not _int_token(toks[0])) or (not _int_token(toks[1])) or (not _int_token(toks[2])):
                new_c_angles.append(raw)
                continue
            i_idx, j_idx, k_idx = int(toks[0]), int(toks[1]), int(toks[2])  # 中文注释：读取数字索引
            ti = idx_to_type.get(i_idx)                                      # 中文注释：索引→类型
            tj = idx_to_type.get(j_idx)
            tk = idx_to_type.get(k_idx)
            if (ti is None) or (tj is None) or (tk is None):
                new_c_angles.append(raw)                    # 中文注释：若无映射则保持原样
                continue
            tail = angles_tail_map.get((ti, tj, tk))        # 中文注释：获取“后四列+注释”
            if tail is None:
                new_c_angles.append(raw)                    # 中文注释：找不到匹配则保持原样
                continue
            # 中文注释：重建行：保留前导空白与前三列索引，用新的尾部（含注释）
            lead = _leading_ws(raw)
            new_line = f"{lead}{toks[0]} {toks[1]} {toks[2]} {tail}"
            if not new_line.endswith("\n"):
                new_line += "\n"
            new_c_angles.append(new_line)
        c_lines[c_angles_s:c_angles_e] = new_c_angles      # 中文注释：切片写回

    # =================== 3) 写出 calculated_{basename} ===================
    out_path = os.path.join(".", f"calculated_{os.path.basename(copolymer_itp_path)}")  # 中文注释：仅用文件名与后缀
    with open(out_path, "w", encoding="utf-8") as fo:
        fo.writelines(c_lines)

    return out_path  # 中文注释：返回输出文件路径

In [ ]:
copolymer_name = df["copolymer_name"][0] # 从df中提取聚合物的名称
new_out_path = apply_calculated_params_to_copolymer(copolymer_itp_path=f"{copolymer_name}.itp", # 聚合物的原始itp文件，力常数来源于GAFF
                                                    replace_fake_copolymer_itp_path=f"replace_fake_{copolymer_name}.itp") # 聚合物的fake_itp文件，大部分键长与键角参数来源于QC计算

In [ ]:
'''
定义文件清扫函数
对于当前目录下，后缀(ext)为下列的文件
    structure_extensions = ('.xyz', '.pdb', '.mol2')
    topology_extensions = ('.top', '.itp', '.chg')
如果文件名不是形如{copolymer_name}.ext或calculated_{copolymer_name}.ext的文件，全部将其移动到“process_documentation”文件夹中，
'''
import os
import shutil
from typing import Tuple

def clean_files(copolymer_name: str,
                structure_extensions: Tuple[str] = ('.xyz', '.pdb', '.mol2'),
                topology_extensions: Tuple[str] = ('.top', '.itp', '.chg'),
                target_dir: str = "process_documentation") -> None:
    """
    清扫当前目录下的多余文件：
    - 如果文件后缀属于 structure_extensions 或 topology_extensions
    - 且文件名不是 {copolymer_name}.ext 或 calculated_{copolymer_name}.ext
    -> 移动到 target_dir 文件夹中

    参数:
    copolymer_name (str): 指定的共聚物名称
    structure_extensions (tuple): 结构文件后缀
    topology_extensions (tuple): 拓扑文件后缀
    target_dir (str): 用于存放无关文件的文件夹
    """
    # 合并所有需要检查的后缀
    valid_extensions = structure_extensions + topology_extensions

    # 确保目标文件夹存在
    os.makedirs(target_dir, exist_ok=True)

    # 遍历当前目录文件
    for file in os.listdir("."):
        if os.path.isfile(file):  # 只处理文件
            _, ext = os.path.splitext(file)

            # 如果文件后缀在目标范围内
            if ext in valid_extensions:
                # 合法文件名1: copolymer_name.ext
                valid_name1 = f"{copolymer_name}{ext}"
                # 合法文件名2: calculated_copolymer_name.ext
                valid_name2 = f"calculated_{copolymer_name}{ext}"

                # 如果不符合合法文件名规则 -> 移动
                if file not in (valid_name1, valid_name2):
                    shutil.move(file, os.path.join(target_dir, file))
                    print(f"Moved: {file} -> {target_dir}/")
                    
copolymer_name = df["copolymer_name"][0] # 从df中提取聚合物的名称
clean_files(copolymer_name)

In [ ]:
import os
import shutil

# 创建目标文件夹（若不存在则创建）
os.makedirs('polymer_structure', exist_ok=True)
os.makedirs('polymer_topology', exist_ok=True)


def _move_file_with_replace(src_file: str, dst_dir: str) -> None:
    """
    中文功能说明：
        将当前目录下生成的结构/拓扑文件移动到目标目录；若目标文件已经存在，先删除旧文件再移动。

    输入参数：
        src_file: 当前目录中的源文件名。
        dst_dir: 目标文件夹，例如 polymer_structure 或 polymer_topology。

    返回值：
        None。

    关键流程：
        1) 拼接目标文件路径；
        2) 如果目标路径已有普通文件或符号链接，先删除；
        3) 调用 shutil.move 完成移动。

    可能报错或边界情况：
        如果目标路径是目录，说明文件名冲突异常，直接抛错，避免误删目录。
    """
    dst_file = os.path.join(dst_dir, os.path.basename(src_file))
    if os.path.isfile(dst_file) or os.path.islink(dst_file):
        os.remove(dst_file)
    elif os.path.isdir(dst_file):
        raise IsADirectoryError(f"目标路径是目录，不能覆盖：{dst_file}")
    shutil.move(src_file, dst_dir)


# 遍历当前目录下的所有文件
for file in os.listdir('.'):
    if file.endswith(('.mol2', '.pdb', '.xyz')):
        _move_file_with_replace(file, 'polymer_structure')
    elif file.endswith(('.itp', '.top', '.chg')):
        _move_file_with_replace(file, 'polymer_topology')

In [ ]:
import os               # 中文注释：操作路径与文件系统
import shutil           # 中文注释：高层次文件与目录拷贝/删除
import zipfile          # 中文注释：创建ZIP压缩包

def collect_and_compress_files(source_dir='.', cleanup=True):
    """
    中文说明：
      - 功能：打包当前目录下的 component_gas 与 dimer_gas 两个文件夹到 all_results/ 下，
              然后将 all_results/ 压缩为 all_results.zip。
      - 参数：
          source_dir (str) : 源目录（默认当前目录 '.'）
          cleanup (bool)   : 压缩完成后是否删除 all_results/ 目录（默认 True，只保留zip）
      - 约定：
          1) 仅处理两个目标目录：component_gas/ 与 dimer_gas/；
          2) 若 all_results/ 已存在则先安全删除，保证结果可重复、无历史残留；
          3) 若目标目录不存在则跳过并给出提示，不中断整体流程。
      - 兼容性：不使用 Python 3.8+ 的 dirs_exist_ok 参数，兼容 Python 3.6+（CentOS7常见）。
    """
    # ---------- 规范化与准备 ----------
    source_dir = os.path.abspath(source_dir)                              # 中文注释：将源目录转为绝对路径，避免相对路径歧义
    all_results_dir = os.path.join(source_dir, 'all_results')             # 中文注释：目标汇总目录
    zip_path = os.path.join(source_dir, 'all_results.zip')                # 中文注释：最终压缩包路径
    targets = ['polymer_structure', 'polymer_topology']                              # 中文注释：仅打包这两个目录

    # ---------- 清理旧的 all_results/ 与旧 zip ----------
    if os.path.isdir(all_results_dir):                                    # 中文注释：若存在历史 all_results/，先删除避免合并脏数据
        shutil.rmtree(all_results_dir)
    os.makedirs(all_results_dir, exist_ok=True)                           # 中文注释：创建全新的 all_results/ 目录
    if os.path.exists(zip_path):                                          # 中文注释：若存在历史压缩包，则先删除，避免覆盖异常
        os.remove(zip_path)

    # ---------- 复制目标目录到 all_results/ ----------
    for name in targets:                                                  # 中文注释：依次处理 component_gas 与 dimer_gas
        src = os.path.join(source_dir, name)                              # 中文注释：源路径
        dst = os.path.join(all_results_dir, name)                         # 中文注释：目标路径（all_results/ 下同名目录）
        if os.path.isdir(src):                                            # 中文注释：仅当源目录存在时才复制
            shutil.copytree(src, dst)                                     # 中文注释：整目录递归复制（Python3.6兼容用法）
        else:
            print(f"[警告] 未找到目录：{src}，已跳过。")                    # 中文注释：缺失时仅提示，不报错

    # ---------- 压缩 all_results/ 为 all_results.zip ----------
    with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:  # 中文注释：创建ZIP（DEFLATED压缩）
        for root, _, files in os.walk(all_results_dir):                   # 中文注释：遍历 all_results/ 下的所有文件
            for f in files:                                               # 中文注释：逐个写入到zip
                fpath = os.path.join(root, f)                             # 中文注释：文件的绝对路径
                arcname = os.path.relpath(fpath, all_results_dir)         # 中文注释：归档名使用相对于 all_results/ 的相对路径
                zf.write(fpath, arcname)                                  # 中文注释：写入压缩包，保持目录结构

    # ---------- 压缩后清理 ----------
    if cleanup:                                                           # 中文注释：若需要“只留压缩包”，删除工作目录
        shutil.rmtree(all_results_dir)

    print(f"[完成] 已生成压缩包：{zip_path}")
    if cleanup:
        print(f"[已清理] 已删除打包目录：{all_results_dir}")

# ===== 使用示例 =====
collect_and_compress_files(source_dir='.', cleanup=True)  # 中文注释：当前目录执行，压缩后删除 all_results/